In [ ]:
# Verify GPU availability

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected.")

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries and load processed datasets

import os
import json
import time
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

LABEL_COLUMNS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

train_df = pd.read_csv(
    os.path.join(PROCESSED_DIR, "train_sub.csv")
)

val_df = pd.read_csv(
    os.path.join(PROCESSED_DIR, "val.csv")
)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# Install transformer training libraries

!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# Load DistilBERT tokenizer

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
# Tokenize training and validation comments

train_encodings = tokenizer(
    train_df["comment_text"].fillna("").tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

val_encodings = tokenizer(
    val_df["comment_text"].fillna("").tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

print("Training tokenized samples:", len(train_encodings["input_ids"]))
print("Validation tokenized samples:", len(val_encodings["input_ids"]))
print("Token sequence length:", len(train_encodings["input_ids"][0]))

In [ ]:
# Create PyTorch datasets for transformer training

from torch.utils.data import Dataset

class ToxicCommentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(
            labels.values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][idx],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][idx],
                dtype=torch.long
            ),
            "labels": self.labels[idx]
        }

train_dataset = ToxicCommentDataset(
    train_encodings, train_df[LABEL_COLUMNS]
)

val_dataset = ToxicCommentDataset(
    val_encodings, val_df[LABEL_COLUMNS]
)

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Number of target labels:", train_dataset[0]["labels"].shape[0])

In [ ]:
# Create DataLoaders for training and validation

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", BATCH_SIZE)

In [ ]:
# Load DistilBERT for six-label classification

from transformers import AutoModelForSequenceClassification

NUM_LABELS = 6

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

model.to("cuda")

print("DistilBERT model loaded successfully.")
print("Number of output labels:", model.config.num_labels)
print("Device:", next(model.parameters()).device)

In [ ]:
# Configure B1 training parameters

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2

print("B1 training configuration")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Warmup ratio:", WARMUP_RATIO)
print("Maximum epochs:", NUM_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

In [ ]:
# Configure AdamW optimizer

from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("AdamW optimizer configured successfully.")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

In [ ]:
# Configure linear learning-rate scheduler

from transformers import get_linear_schedule_with_warmup

TOTAL_TRAINING_STEPS = len(train_loader) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_TRAINING_STEPS * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS
)

print("Scheduler configured successfully.")
print("Total training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

In [ ]:
# Define the training device and initialise the model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    problem_type="multi_label_classification"
).to(device)

print("Random seed:", SEED)
print("Device:", device)
print("Model:", MODEL_NAME)
print("Model device:", next(model.parameters()).device)

In [ ]:
# Recreate AdamW optimizer for the reinitialised model

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("AdamW optimizer recreated successfully.")

In [ ]:
# Recreate scheduler for the current model and optimizer

TOTAL_TRAINING_STEPS = len(train_loader) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_TRAINING_STEPS * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS
)

print("Scheduler recreated successfully.")
print("Total training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

In [ ]:
# Import utilities for B1 training and validation

import os
import json
import time
from sklearn.metrics import average_precision_score
from torch.nn import BCEWithLogitsLoss
from tqdm.auto import tqdm

criterion = BCEWithLogitsLoss()

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "models", "saved_checkpoints",
    "B1_distilbert_bce_best"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("BCE loss configured.")
print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
# Initialise training history and early-stopping variables

best_macro_ap = -1.0
epochs_without_improvement = 0
training_history = []

print("Training tracker initialised.")
print("Best Macro AP:", best_macro_ap)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Maximum epochs:", NUM_EPOCHS)

In [ ]:
# Define one training epoch

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    average_loss = total_loss / len(train_loader.dataset)

    return average_loss


print("Training function defined successfully.")

In [ ]:
# Define validation evaluation

def evaluate_model(model, val_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            probabilities = torch.sigmoid(outputs.logits)

            total_loss += loss.item() * labels.size(0)

            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_labels, axis=0)
    y_scores = np.concatenate(all_probabilities, axis=0)

    average_loss = total_loss / len(val_loader.dataset)
    macro_ap = average_precision_score(
        y_true, y_scores, average="macro"
    )

    return average_loss, macro_ap


print("Validation function defined successfully.")

In [ ]:
# Train and evaluate the first B1 epoch

epoch_start_time = time.time()

train_loss = train_one_epoch(
    model, train_loader, optimizer, scheduler, criterion, device
)

val_loss, val_macro_ap = evaluate_model(
    model, val_loader, criterion, device
)

epoch_time = time.time() - epoch_start_time

print("\nB1 Epoch 1 Results")
print("-------------------------")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Macro AP: {val_macro_ap:.4f}")
print(f"Epoch Time: {epoch_time / 60:.2f} minutes")

In [ ]:
# Record the first epoch results

training_history.append({
    "epoch": 1,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_macro_ap": val_macro_ap,
    "epoch_time_minutes": epoch_time / 60
})

print("Epoch 1 recorded successfully.")
print(training_history)

In [ ]:
# Save the best B1 model checkpoint after Epoch 1

model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

best_macro_ap = float(val_macro_ap)
epochs_without_improvement = 0

print("Best B1 checkpoint saved successfully.")
print(f"Best Validation Macro AP: {best_macro_ap:.4f}")
print("Checkpoint:", CHECKPOINT_DIR)

In [ ]:
# Train and evaluate B1 Epoch 2

epoch_start_time = time.time()

train_loss = train_one_epoch(
    model, train_loader, optimizer, scheduler, criterion, device
)

val_loss, val_macro_ap = evaluate_model(
    model, val_loader, criterion, device
)

epoch_time = time.time() - epoch_start_time

print("\nB1 Epoch 2 Results")
print("-------------------------")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Macro AP: {val_macro_ap:.4f}")
print(f"Epoch Time: {epoch_time / 60:.2f} minutes")

In [ ]:
# Restore B1 training history from completed epoch results

training_history = [
    {
        "epoch": 1,
        "train_loss": 0.1755290716525853,
        "val_loss": 0.0459928433192119,
        "val_macro_ap": 0.7015742399488093,
        "epoch_time_minutes": 17.111273833115895
    },
    {
        "epoch": 2,
        "train_loss": 0.0757,
        "val_loss": 0.0443,
        "val_macro_ap": 0.7326,
        "epoch_time_minutes": 17.02
    }
]

best_macro_ap = 0.7326
epochs_without_improvement = 0

print("Training history restored.")
print("Recorded epochs:", len(training_history))
print("Best Macro AP:", best_macro_ap)

In [ ]:
# Reconnect Google Drive after runtime restart

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore the saved Epoch 1 model from Google Drive

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "models", "saved_checkpoints",
    "B1_distilbert_bce_best"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT_DIR
).to(device)

print("Checkpoint loaded successfully.")
print("Device:", device)
print("Model:", model.config._name_or_path)
print("Number of labels:", model.config.num_labels)

In [ ]:
# Verify the recovered B1 model

print("Model architecture:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("Problem type:", model.config.problem_type)
print("Device:", next(model.parameters()).device)
print("Checkpoint exists:", os.path.exists(CHECKPOINT_DIR))

In [ ]:
# Restore B1 training configuration and datasets

import pandas as pd
import numpy as np
import random
import time
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

LABEL_COLUMNS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_sub.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
MAX_LENGTH = 256
BATCH_SIZE = 16

print("Training environment restored.")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Device:", device)

In [ ]:
# Recreate tokenized datasets and DataLoaders

class ToxicCommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(
            labels, dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }


train_dataset = ToxicCommentDataset(
    train_df["comment_text"].astype(str).tolist(),
    train_df[LABEL_COLUMNS].values,
    tokenizer,
    MAX_LENGTH
)

val_dataset = ToxicCommentDataset(
    val_df["comment_text"].astype(str).tolist(),
    val_df[LABEL_COLUMNS].values,
    tokenizer,
    MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print("DataLoaders restored successfully.")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
# Recreate AdamW optimizer for the recovered model

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = BCEWithLogitsLoss()

print("Optimizer restored.")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

In [ ]:
# Configure scheduler for the remaining three epochs

REMAINING_EPOCHS = 3
TOTAL_TRAINING_STEPS = len(train_loader) * REMAINING_EPOCHS
WARMUP_STEPS = int(TOTAL_TRAINING_STEPS * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS
)

print("Recovery scheduler configured.")
print("Remaining epochs:", REMAINING_EPOCHS)
print("Total training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

In [ ]:
# Restore the B1 training function

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(train_loader.dataset)


print("B1 training function restored successfully.")

In [ ]:
# Restore B1 validation function

def evaluate_model(model, val_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            probabilities = torch.sigmoid(outputs.logits)

            total_loss += loss.item() * labels.size(0)
            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_labels, axis=0)
    y_scores = np.concatenate(all_probabilities, axis=0)

    average_loss = total_loss / len(val_loader.dataset)
    macro_ap = average_precision_score(
        y_true, y_scores, average="macro"
    )

    return average_loss, macro_ap


print("B1 validation function restored successfully.")


In [ ]:
# Restore required Python and evaluation utilities

import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from transformers import get_linear_schedule_with_warmup

print("All required utilities imported successfully.")
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Complete B1 recovery after Colab runtime restart

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

# Restore project configuration
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "models", "saved_checkpoints",
    "B1_distilbert_bce_best"
)

LABEL_COLUMNS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
REMAINING_EPOCHS = 3

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load saved Epoch 1 checkpoint
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT_DIR
).to(device)

# Load datasets
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_sub.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))

# Recreate datasets
class ToxicCommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

train_dataset = ToxicCommentDataset(
    train_df["comment_text"].astype(str).tolist(),
    train_df[LABEL_COLUMNS].values,
    tokenizer, MAX_LENGTH
)

val_dataset = ToxicCommentDataset(
    val_df["comment_text"].astype(str).tolist(),
    val_df[LABEL_COLUMNS].values,
    tokenizer, MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

# Recreate optimizer, scheduler and loss
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = BCEWithLogitsLoss()

TOTAL_TRAINING_STEPS = len(train_loader) * REMAINING_EPOCHS
WARMUP_STEPS = int(TOTAL_TRAINING_STEPS * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS
)

print("\nB1 recovery completed.")
print("Model:", model.__class__.__name__)
print("Device:", device)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Optimizer and scheduler: Ready")
print("Saved checkpoint: Epoch 1")

In [ ]:
# Restore B1 training and validation functions

def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(train_loader.dataset)


def evaluate_model(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            probabilities = torch.sigmoid(outputs.logits)

            total_loss += loss.item() * labels.size(0)
            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_labels, axis=0)
    y_scores = np.concatenate(all_probabilities, axis=0)

    average_loss = total_loss / len(val_loader.dataset)
    macro_ap = average_precision_score(
        y_true, y_scores, average="macro"
    )

    return average_loss, macro_ap


print("B1 training and validation functions restored.")

In [ ]:
# Repeat B1 Epoch 2 and evaluate

epoch_start_time = time.time()

train_loss, val_loss, val_macro_ap = None, None, None

train_loss = train_one_epoch(
    model, train_loader, optimizer, scheduler, criterion, device
)

val_loss, val_macro_ap = evaluate_model(
    model, val_loader, criterion, device
)

epoch_time = time.time() - epoch_start_time

print("\nB1 Recovery Epoch 2 Results")
print("----------------------------")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Macro AP: {val_macro_ap:.4f}")
print(f"Epoch Time: {epoch_time / 60:.2f} minutes")

In [ ]:
# Save improved B1 checkpoint and recovery state

model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

torch.save({
    "epoch": 2,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_macro_ap": float(val_macro_ap),
    "epoch_time_minutes": epoch_time / 60,
    "seed": SEED,
    "recovery_run": True
}, os.path.join(CHECKPOINT_DIR, "training_state.pt"))

print("B1 recovery checkpoint saved successfully.")
print(f"Best Validation Macro AP: {val_macro_ap:.4f}")
print("Model, optimizer and scheduler states saved.")

In [ ]:
# Train and evaluate B1 Epoch 3

epoch_start_time = time.time()

train_loss = train_one_epoch(
    model, train_loader, optimizer, scheduler, criterion, device
)

val_loss, val_macro_ap = evaluate_model(
    model, val_loader, criterion, device
)

epoch_time = time.time() - epoch_start_time

print("\nB1 Epoch 3 Results")
print("-------------------------")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Macro AP: {val_macro_ap:.4f}")
print(f"Epoch Time: {epoch_time / 60:.2f} minutes")

In [ ]:
# Save the improved B1 Epoch 3 checkpoint

model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

torch.save({
    "epoch": 3,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_macro_ap": float(val_macro_ap),
    "epoch_time_minutes": epoch_time / 60,
    "seed": SEED,
    "recovery_run": True
}, os.path.join(CHECKPOINT_DIR, "training_state.pt"))

best_macro_ap = float(val_macro_ap)
epochs_without_improvement = 0

print("Epoch 3 checkpoint saved successfully.")
print("Best Validation Macro AP:", best_macro_ap)

In [ ]:
# Save the improved B1 Epoch 3 checkpoint

model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

torch.save({
    "epoch": 3,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_macro_ap": float(val_macro_ap),
    "epoch_time_minutes": epoch_time / 60,
    "seed": SEED,
    "recovery_run": True
}, os.path.join(CHECKPOINT_DIR, "training_state.pt"))

best_macro_ap = float(val_macro_ap)
epochs_without_improvement = 0

print("Epoch 3 checkpoint saved successfully.")
print("Best Validation Macro AP:", best_macro_ap)

In [ ]:
# Train and evaluate B1 Epoch 4

epoch_start_time = time.time()

train_loss = train_one_epoch(
    model, train_loader, optimizer, scheduler, criterion, device
)

val_loss, val_macro_ap = evaluate_model(
    model, val_loader, criterion, device
)

epoch_time = time.time() - epoch_start_time

print("\nB1 Epoch 4 Results")
print("-------------------------")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Macro AP: {val_macro_ap:.4f}")
print(f"Epoch Time: {epoch_time / 60:.2f} minutes")

In [ ]:
# Save B1 experiment results and recovery details

b1_log = {
    "experiment": "B1",
    "model": "distilbert-base-uncased",
    "loss": "Binary Cross Entropy",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "selected_epoch": 3,
    "best_validation_macro_ap": 0.7307863675,
    "checkpoint": CHECKPOINT_DIR,
    "runtime_recovery": True,
    "epochs": [
        {
            "epoch": 1,
            "train_loss": 0.175529,
            "val_loss": 0.045993,
            "macro_ap": 0.701574
        },
        {
            "epoch": 2,
            "train_loss": 0.0940,
            "val_loss": 0.0591,
            "macro_ap": 0.7234,
            "recovery": True
        },
        {
            "epoch": 3,
            "train_loss": 0.0769,
            "val_loss": 0.0440,
            "macro_ap": 0.730786,
            "recovery": True
        },
        {
            "epoch": 4,
            "train_loss": 0.0601,
            "val_loss": 0.0471,
            "macro_ap": 0.7246,
            "recovery": True
        }
    ],
    "notes": (
        "Original Epoch 2 achieved Macro AP 0.7326, "
        "but its weights were not saved after runtime restart. "
        "Recovery training resumed from the saved Epoch 1 checkpoint "
        "with a newly initialised optimizer and scheduler. "
        "Epoch 3 is the best reproducible saved checkpoint."
    )
}

log_path = os.path.join(
    PROJECT_DIR,
    "results",
    "experiment_logs",
    "B1_distilbert_bce.json"
)

os.makedirs(os.path.dirname(log_path), exist_ok=True)

with open(log_path, "w") as f:
    json.dump(b1_log, f, indent=4)

print("B1 experiment log saved successfully.")
print("Selected epoch:", b1_log["selected_epoch"])
print("Best Macro AP:", b1_log["best_validation_macro_ap"])
print("Log:", log_path)

In [ ]:
## B2 — RoBERTa + Binary Cross-Entropy Loss

In [ ]:
# Load RoBERTa tokenizer for B2

from transformers import AutoTokenizer

B2_MODEL_NAME = "roberta-base"
B2_MAX_LENGTH = 256

b2_tokenizer = AutoTokenizer.from_pretrained(B2_MODEL_NAME)

print("B2 tokenizer loaded successfully.")
print("Model:", B2_MODEL_NAME)
print("Vocabulary size:", b2_tokenizer.vocab_size)
print("Maximum sequence length:", B2_MAX_LENGTH)

In [ ]:
# Tokenize B2 training and validation data

b2_train_encodings = b2_tokenizer(
    train_df["comment_text"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=B2_MAX_LENGTH,
    return_tensors="pt"
)

b2_val_encodings = b2_tokenizer(
    val_df["comment_text"].astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=B2_MAX_LENGTH,
    return_tensors="pt"
)

print("B2 tokenization completed.")
print("Training samples:", b2_train_encodings["input_ids"].shape)
print("Validation samples:", b2_val_encodings["input_ids"].shape)

In [ ]:
# Create B2 PyTorch datasets

class RoBERTaToxicDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(
            labels, dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }


b2_train_dataset = RoBERTaToxicDataset(
    b2_train_encodings,
    train_df[LABEL_COLUMNS].values
)

b2_val_dataset = RoBERTaToxicDataset(
    b2_val_encodings,
    val_df[LABEL_COLUMNS].values
)

print("B2 datasets created successfully.")
print("Training samples:", len(b2_train_dataset))
print("Validation samples:", len(b2_val_dataset))
print("Number of labels:", b2_train_dataset[0]["labels"].shape[0])

In [ ]:
# Create B2 training and validation DataLoaders

B2_BATCH_SIZE = 16

b2_train_loader = DataLoader(
    b2_train_dataset,
    batch_size=B2_BATCH_SIZE,
    shuffle=True
)

b2_val_loader = DataLoader(
    b2_val_dataset,
    batch_size=B2_BATCH_SIZE,
    shuffle=False
)

print("B2 DataLoaders created successfully.")
print("Training batches:", len(b2_train_loader))
print("Validation batches:", len(b2_val_loader))
print("Batch size:", B2_BATCH_SIZE)

In [ ]:
# Load RoBERTa for six-label classification

from transformers import AutoModelForSequenceClassification

b2_model = AutoModelForSequenceClassification.from_pretrained(
    B2_MODEL_NAME,
    num_labels=6,
    problem_type="multi_label_classification"
).to(device)

print("B2 model loaded successfully.")
print("Architecture:", b2_model.__class__.__name__)
print("Number of labels:", b2_model.config.num_labels)
print("Device:", next(b2_model.parameters()).device)

In [ ]:
# Configure B2 training parameters

B2_LEARNING_RATE = 2e-5
B2_WEIGHT_DECAY = 0.01
B2_WARMUP_RATIO = 0.10
B2_NUM_EPOCHS = 4
B2_EARLY_STOPPING_PATIENCE = 2

print("B2 training configuration")
print("Learning rate:", B2_LEARNING_RATE)
print("Weight decay:", B2_WEIGHT_DECAY)
print("Warmup ratio:", B2_WARMUP_RATIO)
print("Maximum epochs:", B2_NUM_EPOCHS)
print("Early stopping patience:", B2_EARLY_STOPPING_PATIENCE)

In [ ]:
# Configure AdamW optimizer for B2

b2_optimizer = AdamW(
    b2_model.parameters(),
    lr=B2_LEARNING_RATE,
    weight_decay=B2_WEIGHT_DECAY
)

print("B2 AdamW optimizer configured successfully.")
print("Learning rate:", B2_LEARNING_RATE)
print("Weight decay:", B2_WEIGHT_DECAY)

In [ ]:
# Configure B2 linear learning-rate scheduler

B2_TOTAL_STEPS = len(b2_train_loader) * B2_NUM_EPOCHS
B2_WARMUP_STEPS = int(B2_TOTAL_STEPS * B2_WARMUP_RATIO)

b2_scheduler = get_linear_schedule_with_warmup(
    b2_optimizer,
    num_warmup_steps=B2_WARMUP_STEPS,
    num_training_steps=B2_TOTAL_STEPS
)

print("B2 scheduler configured successfully.")
print("Total training steps:", B2_TOTAL_STEPS)
print("Warmup steps:", B2_WARMUP_STEPS)

In [ ]:
# Configure binary cross-entropy loss for B2

b2_criterion = torch.nn.BCEWithLogitsLoss()

print("B2 BCE loss configured successfully.")

In [ ]:
# Define B2 training and validation functions

import numpy as np
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

def b2_train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="B2 Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def b2_evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="B2 Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            probabilities = torch.sigmoid(outputs.logits)

            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.vstack(all_labels)
    y_prob = np.vstack(all_probabilities)

    macro_ap = average_precision_score(
        y_true,
        y_prob,
        average="macro"
    )

    return total_loss / len(loader), macro_ap


print("B2 training and validation functions defined successfully.")

In [ ]:
# Locate the existing project folder in Google Drive

import os

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"

if not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}")

print("Project folder verified:", PROJECT_ROOT)
print("Folder exists:", os.path.isdir(PROJECT_ROOT))

In [ ]:
# Set up B2 checkpoint and experiment-log paths

import os
import json

B2_CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT, "models", "saved_checkpoints", "B2_roberta_bce_best"
)

B2_LATEST_STATE_PATH = os.path.join(
    PROJECT_ROOT, "models", "saved_checkpoints",
    "B2_roberta_bce_training_state.pt"
)

B2_LOG_PATH = os.path.join(
    PROJECT_ROOT, "results", "experiment_logs",
    "B2_roberta_bce.json"
)

os.makedirs(B2_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(B2_LOG_PATH), exist_ok=True)

b2_history = []
b2_best_macro_ap = 0.0
b2_best_epoch = 0
b2_epochs_without_improvement = 0

print("B2 checkpoint and log paths configured successfully.")
print("Best model:", B2_CHECKPOINT_DIR)
print("Latest training state:", B2_LATEST_STATE_PATH)
print("Experiment log:", B2_LOG_PATH)

In [ ]:
# Initialize B2 experiment metadata

b2_config = {
    "experiment": "B2_RoBERTa_base_BCE",
    "model": B2_MODEL_NAME,
    "max_length": B2_MAX_LENGTH,
    "batch_size": 16,
    "learning_rate": B2_LEARNING_RATE,
    "weight_decay": B2_WEIGHT_DECAY,
    "warmup_ratio": B2_WARMUP_RATIO,
    "max_epochs": B2_NUM_EPOCHS,
    "early_stopping_patience": B2_EARLY_STOPPING_PATIENCE,
    "primary_metric": "Validation Macro Average Precision",
    "loss": "BCEWithLogitsLoss"
}

print("B2 experiment metadata initialized.")
print(json.dumps(b2_config, indent=4))

In [ ]:
# Train B2 and save recovery checkpoints after every epoch

import torch
from transformers import AutoTokenizer

b2_tokenizer = AutoTokenizer.from_pretrained(B2_MODEL_NAME)

for b2_epoch in range(1, B2_NUM_EPOCHS + 1):

    print(f"\n{'='*55}")
    print(f"B2 — EPOCH {b2_epoch}/{B2_NUM_EPOCHS}")
    print(f"{'='*55}")

    b2_train_loss = b2_train_one_epoch(
        b2_model,
        b2_train_loader,
        b2_optimizer,
        b2_scheduler,
        b2_criterion
    )

    b2_val_loss, b2_macro_ap = b2_evaluate_model(
        b2_model,
        b2_val_loader,
        b2_criterion
    )

    b2_improved = b2_macro_ap > b2_best_macro_ap

    if b2_improved:
        b2_best_macro_ap = b2_macro_ap
        b2_best_epoch = b2_epoch
        b2_epochs_without_improvement = 0

        b2_model.save_pretrained(B2_CHECKPOINT_DIR)
        b2_tokenizer.save_pretrained(B2_CHECKPOINT_DIR)
    else:
        b2_epochs_without_improvement += 1

    b2_epoch_record = {
        "epoch": b2_epoch,
        "train_loss": float(b2_train_loss),
        "validation_loss": float(b2_val_loss),
        "validation_macro_ap": float(b2_macro_ap),
        "best_checkpoint_updated": bool(b2_improved)
    }

    b2_history.append(b2_epoch_record)

    # Save the latest state for recovery
    torch.save({
        "epoch": b2_epoch,
        "model_state_dict": b2_model.state_dict(),
        "optimizer_state_dict": b2_optimizer.state_dict(),
        "scheduler_state_dict": b2_scheduler.state_dict(),
        "history": b2_history,
        "best_macro_ap": b2_best_macro_ap,
        "best_epoch": b2_best_epoch,
        "epochs_without_improvement": b2_epochs_without_improvement,
        "config": b2_config
    }, B2_LATEST_STATE_PATH)

    with open(B2_LOG_PATH, "w") as f:
        json.dump({
            "config": b2_config,
            "history": b2_history,
            "best_epoch": b2_best_epoch,
            "best_validation_macro_ap": b2_best_macro_ap
        }, f, indent=4)

    print(f"\nTrain Loss: {b2_train_loss:.4f}")
    print(f"Validation Loss: {b2_val_loss:.4f}")
    print(f"Validation Macro AP: {b2_macro_ap:.4f}")
    print(f"Best Macro AP: {b2_best_macro_ap:.4f} (Epoch {b2_best_epoch})")
    print("Checkpoint and experiment log saved.")

    if b2_epochs_without_improvement >= B2_EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nB2 training loop completed.")
print("Best epoch:", b2_best_epoch)
print("Best validation Macro AP:", round(b2_best_macro_ap, 4))